# MMMU 오염 정식 검사 (Colab)

`reports/dataset_search.md` 2장 절차를 그대로 돌립니다(SSCD 이미지 유사도, 13-gram, 다국어 임베딩 + LLM 판정, recall 측정).

- 런타임 → 런타임 유형 변경 → **GPU** (T4면 충분, L4·A100이면 더 빠름)
- 결과와 E 인덱스는 **Google Drive `MyDrive/decontam/`**에 쌓입니다. 세션이 끊기면 1번부터 다시 실행하면 되고, 이미 끝난 단계는 건너뜁니다.
- 탭을 닫으면 세션이 끊길 수 있으니 열어 두세요.

In [ ]:
# 1. GPU 확인 + Drive 연결
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 코드 받기 (reports 브랜치) + 패키지
%cd /content
!test -d MMDL-MMMU || git clone -q -b reports https://github.com/jinyoung924/MMDL-MMMU.git
%cd /content/MMDL-MMMU
!git pull -q
!git log --oneline -1
!pip install -q gdown pyarrow

import os
os.environ['WORK'] = '/content/drive/MyDrive/decontam'      # E 인덱스·recall·결과 (끊겨도 남음)
os.environ['DATA'] = '/content/data'                         # 데이터셋 원본 (로컬 디스크, 끊기면 다시 받음)
os.environ['HF_HOME'] = '/content/hf'
os.environ['OUT'] = '/content/drive/MyDrive/decontam/out'
os.environ['PYTHONUNBUFFERED'] = '1'
os.makedirs(os.environ['OUT'], exist_ok=True)

## 3. 확인 실행: SLAKE 하나

E 인덱스(MMMU·Pro 이미지 임베딩)와 recall 측정을 먼저 만들고 SLAKE를 검사합니다. 처음 한 번은 모델·데이터 다운로드 때문에 오래 걸립니다.
마지막에 `[run_all ...] done`이 나오고 4번 셀에서 slake 줄이 보이면 성공입니다.

In [ ]:
!DATASETS=slake bash decontam/run_all.sh "$OUT" 2>&1 | tee -a "$WORK/log_slake.txt"

In [ ]:
# 4. 결과 보기 (데이터셋별 4열 한 줄 + 뺄 레코드 수)
import json, glob, os
for f in sorted(glob.glob(os.environ['OUT'] + '/*/report.json')):
    r = json.load(open(f))
    n = sum(1 for _ in open(f.replace('report.json', 'removed_ids.txt'))) if os.path.exists(f.replace('report.json', 'removed_ids.txt')) else 0
    print(f.split('/')[-2], '|', r.get('line'), '| 뺄 레코드', n)
rec = os.environ['WORK'] + '/recall/recall.json'
if os.path.exists(rec): print('recall:', json.load(open(rec)))

## 5. 전체 실행

3번이 성공했으면 나머지 13개를 돌립니다. SLAKE는 이미 끝났으니 건너뜁니다. 끊기면 1·2번을 다시 실행한 뒤 이 셀만 다시 돌리면 끝난 데이터셋을 건너뛰고 이어서 합니다.
끝나면 4번 셀을 다시 실행하고, `MyDrive/decontam/out/` 폴더(특히 `dataset_search_filled.md`와 각 `removed_ids.txt`)를 받아 주세요.

In [ ]:
!bash decontam/run_all.sh "$OUT" 2>&1 | tee -a "$WORK/log_all.txt"